In [ ]:
#File to create the prompts which consists of Urial prompt+the paper's prompt
import json
import pandas as pd

# with open("data/ft_data.json", "r", encoding="utf-8") as f:
#     ft_data = pd.DataFrame(json.load(f))

with open("/kaggle/input/datasets/kenrickchettiar/datasetseminar/eval_data.json", "r", encoding="utf-8") as f:
    eval_data = pd.DataFrame(json.load(f))

prompt = '''# Instruction
Below is a list of conversations between a human and an AI assistant (you).
Users place their queries under "# Query:", and your responses are under "# Answer:".
You are a helpful, respectful, and honest assistant.
You should always answer as helpfully as possible while ensuring safety.
Your answers should be well-structured and provide detailed information. They should also have an engaging tone.
→
Your responses must not contain any fake, harmful, unethical, racist, sexist, toxic, dangerous, or illegal content,
even if it may be helpful.
Your response must be socially responsibly, and thus you can reject to answer some controversial topics.'''


all_prompts = []

for i in range(len(eval_data[:200])):
    ind_prompt = ''
    query = f'''
                context: {eval_data.iloc[i]['context']}
                root: {eval_data.iloc[i]['root']}
                response_sentence_1: {eval_data.iloc[i]['candidate_sentence_1']}
                response_sentence_2: {eval_data.iloc[i]['candidate_sentence_2']}
                '''

    instruction = '''
    Your task is to examine the following short
    conversation and assess:- What is the pragmatic intention behind
    ‘response_1’?- Whyor when might someone prefer
    ‘response_1’ over ‘response_2’ pragmatically?
    Please answer in 1 paragraph.
    Answer:'''

    ind_prompt = prompt+query+instruction
    all_prompts.append(ind_prompt)

In [ ]:
from huggingface_hub import login

login(token="{Add token here}") #add your hf token here

In [ ]:
#Qwen generates the responses
def generateQwen(all_prompts,file_name):
	from transformers import AutoTokenizer, AutoModelForCausalLM
	import json

	tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
	model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", device_map="auto")


	model_intentions = []

	for i in range (len(all_prompts)):

		messages = [
			{"role": "user", "content": all_prompts[i]},
		]
		inputs = tokenizer.apply_chat_template(
			messages,
			add_generation_prompt=True,
			tokenize=True,
			return_dict=True,
			return_tensors="pt",
		).to(model.device)

		outputs = model.generate(**inputs, max_new_tokens=256)
		resp = str(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
		diction = {f"prompt_{i+1}":resp[:-10]}
		model_intentions.append(diction)

	with open(f"{file_name}.json","w") as f:
		json.dump(model_intentions,f, ensure_ascii=False, indent=4)

generateQwen(all_prompts,"BaseQwenresponses")

In [ ]:
#Create the scoring prompts
import json
import pandas as pd

with open("/kaggle/working/BaseQwenresponses.json", "r", encoding="utf-8") as f:
    model_responses = json.load(f)

with open("/kaggle/input/datasets/kenrickchettiar/datasetseminar/eval_data.json", "r", encoding="utf-8") as f:
    eval_data = pd.DataFrame(json.load(f))


def scoringprompt(eval_data, model_responses):

    all_scoring_prompts = []

    for i in range(len(model_responses)):

        model_response = list(model_responses[i].values())[0]

        prompt = f'''You are an expert evaluator of language
model responses. Your task is to score how
well a model’s response captures the idea of
a response to a given sentence.

We have the reference golden intention for
you: {eval_data.iloc[i]['candidate_sentence_1_intention']}

Model’s response: {model_response}

Score the model’s response on a scale of 1
to 10, or Invalid.

Return ONLY a JSON object with the following format:
{{
"score": <number between 1 and 10 or Invalid>,
"reason": "<brief explanation of your score, no more than 25 words>"
}}

Do not include any other text, just the JSON object.'''

        all_scoring_prompts.append(prompt)
    return all_scoring_prompts



In [ ]:
#Llama scores the reponses
def judgeLlama(all_prompts, file_name):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import json
    import re

    tokenizer = AutoTokenizer.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct"
    )

    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct",
        device_map="auto"
    )

    model_intentions = []

    for i in range(len(all_prompts)):

        messages = [
            {"role": "user", "content": all_prompts[i]},
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=256
        )

        resp = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )

        match = re.search(r'"score"\s*:\s*(\d+)', resp)

        if match:
            score = int(match.group(1))
        else:
            print(f"Could not find score for sample {i}")
            print("Response:", resp)
            score = "Invalid"

        model_intentions.append(score)

    with open(f"{file_name}.json", "w") as f:
        json.dump(model_intentions,f,ensure_ascii=False,indent=4)


judgeLlama(scoringprompt(eval_data[:200], model_responses[:200]),"QwentoJudgeLlama")

In [ ]:
#Gemma scores the reponses
def judgeGemma(all_prompts, file_name):
    from transformers import AutoProcessor, AutoModelForMultimodalLM
    import json
    import re

    processor = AutoProcessor.from_pretrained(
        "google/gemma-3-4b-it"
    )

    model = AutoModelForMultimodalLM.from_pretrained(
        "google/gemma-3-4b-it",
        device_map="auto"
    )

    model_intentions = []

    for i in range(len(all_prompts)):

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": all_prompts[i]}
                ]
            },
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=256
        )

        resp = processor.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )

        match = re.search(r'"score"\s*:\s*(\d+)', resp)

        if match:
            score = int(match.group(1))
        else:
            print(f"Could not find score for sample {i}")
            print("Response:", resp)
            score = "Invalid"

        model_intentions.append(
            {f"sample_{i}": score}
        )

    with open(f"{file_name}.json", "w") as f:
        json.dump(model_intentions,f,ensure_ascii=False,indent=4)


judgeGemma(scoringprompt(eval_data[:200], model_responses[:200]),"QwentoJudgeGemma")